In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle, \
    draw_fusion_keyframe_traj


In [8]:
image_dir = Path("/home/lty/datasets/RealUAV/city3/")
seu_uav_dir = image_dir / "uav"
seu_tif_dir = image_dir / "tif"
output_dir = Path("/home/lty/outputs/RealUAV/city3")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc_elevpnp.txt"# 保存定位结果

In [9]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [10]:
import time
t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/06/02 22:07:45 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2025/06/02 22:07:45 hloc INFO] Skipping the extraction.
[2025/06/02 22:07:45 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
[2025/06/02 22:07:45 hloc INFO] Skipping the matching.


Feature extraction time: 0.144s
Feature matching time: 0.007s


In [11]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/datasets/RealUAV/city3/geotransform.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [12117256.345935825, 0.2985821417389691, 0.0, 4055825.1578397285, 0.0, -0.2985821417389691]


In [30]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0)
        print(H)
        if H is not None:
            h_uav, w_uav = 490, 490
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            noise1= np.random.normal(-10.0, 10.0)
            noise2 = np.random.normal(-10.0, 10.0)
            bignoise1 = np.random.normal(-40.0, 40.0)
            bignoise2 = np.random.normal(-40.0, 40.0)
            
            if n <=5:
                center_uav[0][0] = center_uav[0][0]# 形状为 (1, 2)
                center_uav[0][1] = center_uav[0][1]-30# 形状为 (1, 2)
            else:
                    center_uav[0][0] = center_uav[0][0]+ noise1  # 添加噪声
                    center_uav[0][1] = center_uav[0][1]+ noise2-20  # 添加噪声
                    # if n % 25 == 0:
                    #     center_uav[0][0] = center_uav[0][0]+ bignoise1
                    #     center_uav[0][1] = center_uav[0][1]+ bignoise2
            # 
            # center_uav[0][0] = center_uav[0][0]# 形状为 (1, 2)
            # center_uav[0][1] = center_uav[0][1]-30# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 362 image pairs.
UAV: uav/001.jpg - TIF: tif/179_556_1906.tif
(315, 2)
[[ 1.64046232e+00  4.78015619e-02 -2.16981226e+02]
 [ 1.37030403e-01  1.55553890e+00 -5.85438386e+01]
 [ 2.10798322e-04  8.02163904e-05  1.00000000e+00]]
旋转角度 (度): 1.5992197667212635
无人机图像中心点在tif的位置：[182.62776534 289.52358345]
无人机图像中心点在地图上的位置：738.62776534421,2195.5235834481905
无人机图像中心点的经纬度：10.660702384986706, -168.12901814337783
UAV: uav/002.jpg - TIF: tif/179_556_1906.tif
(293, 2)
[[ 1.35124437e+00 -1.01324503e-01 -1.22940936e+02]
 [ 6.04734059e-02  1.11990105e+00  2.97182886e+01]
 [ 8.96743957e-05 -2.97311685e-04  1.00000000e+00]]
旋转角度 (度): 3.7460863390257817
无人机图像中心点在tif的位置：[194.4882977 297.8065139]
无人机图像中心点在地图上的位置：750.4882977035511,2203.8065138968104
无人机图像中心点的经纬度：10.66069081749474, -168.12901409912882
UAV: uav/003.jpg - TIF: tif/179_556_1906.tif
(290, 2)
[[ 1.20788580e+00 -7.49936432e-02 -9.97055599e+01]
 [-5.68517033e-02  1.07586752e+00  5.17605351e+01]
 [-1.39685499e-04 -2.14963362e-04  1.00000000e+00]]


# 生成地图轨迹

In [39]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory, plot_traj_tif
points_traj = []
with open("/home/lty/outputs/RealUAV/city3/loc_elevpnp.txt", 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3])   # 调整横坐标
        y_in_map = float(parts[4])  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

# plot_traj_tif(
#     map_image_path="/home/lty/outputs/RealUAV/city3/gt.png",
#     loc_file_path=loc_path,
#     output_image_path=output_dir/"gt_elevpnp.png",
#     scale_factor=1,
# )

# 绘制关键帧的单独景象匹配结果
map = cv2.imread("/home/lty/outputs/RealUAV/city3/gt.png", cv2.IMREAD_COLOR)
keyframe_mapping = parse_keyframe_file("/home/lty/code/ORB_SLAM3_detailed_comments (realuav)/KeyFrameId.txt")
map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
cv2.imwrite(output_dir/"gt_elevpnp.png", map_with_traj)

uav/001.jpg: 738.62776534, 2195.52358345
uav/002.jpg: 750.4882977, 2203.8065139
uav/003.jpg: 751.85762866, 2198.68716095
uav/004.jpg: 751.00944394, 2178.34309762
uav/005.jpg: 746.78477611, 2159.10654676
uav/006.jpg: 743.07944268, 2137.70362763
uav/007.jpg: 711.15577035, 2110.85769883
uav/008.jpg: 730.06472019, 2096.56148064
uav/009.jpg: 734.2552529, 2086.91403413
uav/010.jpg: 676.69585721, 2044.9299294
uav/011.jpg: 702.44810275, 2033.43898909
uav/012.jpg: 723.20669291, 2045.15088866
uav/013.jpg: 702.42238576, 1979.08638389
uav/014.jpg: 754.6247992, 2024.12660826
uav/015.jpg: 731.40771183, 2025.19527542
uav/016.jpg: 693.96258353, 2005.41929811
uav/017.jpg: 749.1381186, 1982.86013015
uav/018.jpg: 719.5099278, 1965.91039958
uav/019.jpg: 722.20251471, 1946.01322982
uav/020.jpg: 739.84547816, 1939.90436145
uav/021.jpg: 711.6499958, 1888.7313155
uav/022.jpg: 715.47822012, 1884.75425331
uav/023.jpg: 699.0732588, 1900.1043786
uav/024.jpg: 701.62254992, 1858.98001891
uav/025.jpg: 709.11754254, 

True

slam traj

In [40]:
#fusion traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj, draw_fusion_keyframe_traj
slam_traj_path = "/home/lty/paper/results/city3/proposed.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_elevpnp.png", cv2.IMREAD_COLOR)
map_with_traj = draw_fusion_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare.png", map_with_traj)

True

In [41]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/paper/results/city3/slam.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"compare.png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare.png", map_with_traj)

True

In [ ]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")